In [51]:
import os
import pandas as pd


In [52]:
catalogues = {
    "A": "../data/raw/catalogue/jwst_bubble_properties_A.txt",
    "B": "../data/raw/catalogue/jwst_bubble_properties_B_fixed.txt",
    "C": "../data/raw/catalogue/jwst_bubble_properties_C.txt",
}

bad_bubbles_path = "../data/processed/metadata/bad_bubble_id.txt"

In [53]:
all_catalogues = []

for catalogue_label, path in catalogues.items():
    df = pd.read_csv(path)
    df["CATALOGUE"] = catalogue_label
    all_catalogues.append(df)

df_all = pd.concat(all_catalogues, ignore_index=True)

# Creating a global unique ID
df_all.insert(0, "GLOBAL_ID", range(1, len(df_all) + 1))

print("Total bubbles before cleaning:", len(df_all))
df_all[["GLOBAL_ID", "CATALOGUE", "ID", "AVG_RAD_PC"]].head()

Total bubbles before cleaning: 3318


,GLOBAL_ID,CATALOGUE,ID,AVG_RAD_PC
0,1,A,1,86
1,2,A,2,24
2,3,A,3,42
3,4,A,4,25
4,5,A,5,24


In [54]:
bad_df = pd.read_csv(bad_bubbles_path)

print("Bad bubbles listed:", len(bad_df))
bad_df.head()

Bad bubbles listed: 78


,CATALOGUE,ID
0,A,1506
1,A,1450
2,A,1161
3,A,1222
4,A,1332


In [55]:
# df housekeeping

df_all.columns = df_all.columns.str.strip()
df_all["CATALOGUE"] = df_all["CATALOGUE"].astype(str).str.strip()
df_all["ID"] = df_all["ID"].astype(int)


bad_df.columns = bad_df.columns.str.strip()
bad_df["CATALOGUE"] = bad_df["CATALOGUE"].astype(str).str.strip()
bad_df["ID"] = bad_df["ID"].astype(int)

bad_df_unique = bad_df.drop_duplicates(subset=["CATALOGUE", "ID"])

print("Bad bubbles listed:", len(bad_df))
print("Unique bad bubbles:", len(bad_df_unique))

# checking bad bubbles matching df_all
matched_bad = bad_df_unique.merge(
    df_all[["CATALOGUE", "ID"]],
    on=["CATALOGUE", "ID"],
    how="inner"
)

unmatched_bad = bad_df_unique.merge(
    df_all[["CATALOGUE", "ID"]],
    on=["CATALOGUE", "ID"],
    how="left",
    indicator=True
)

unmatched_bad = unmatched_bad[unmatched_bad["_merge"] == "left_only"]

print("Matching bad bubbles:", len(matched_bad))
print("Unmatched bad bubbles:", len(unmatched_bad))

display(unmatched_bad)


Bad bubbles listed: 78
Unique bad bubbles: 60
Matching bad bubbles: 59
Unmatched bad bubbles: 1


,CATALOGUE,ID,_merge
38,A,1698,left_only


In [56]:
df_clean = df_all.merge(
    bad_df_unique[["CATALOGUE", "ID"]],
    on=["CATALOGUE", "ID"],
    how="left",
    indicator=True
)

df_clean = df_clean[df_clean["_merge"] == "left_only"].drop(columns=["_merge"])

print("Total bubbles after cleaning:", len(df_clean))
print("Removed bubbles:", len(df_all) - len(df_clean))
print("In bad bubbles file", len(bad_df))

Total bubbles after cleaning: 3259
Removed bubbles: 59
In bad bubbles file 78


In [57]:
collective_path = "../data/processed/metadata/catalogue_collective.txt"

df_clean.to_csv(collective_path, index=False)

print("Saved full collective catalogue to", collective_path)

Saved full collective catalogue to ../data/processed/metadata/catalogue_collective.txt
